# Build
With Generative AI and RAG specifically, most of the 'featurization' of our data is simply preparing a vector search index. This notebook takes the processed chunks (which contain tables, images, pages, and text chunks). It has been tested with Serverless v3.

In [0]:
%pip install uv

In [0]:
%sh uv pip install .

In [0]:
%restart_python

In [0]:
import sys

sys.path.append(".")

## Move converted index to a table

Because our conversion process overtook spark, we reload the cached chunks.parquet file and save it to a table

In [0]:
import mlflow
from mlflow.models import ModelConfig
from src.agent.config import parse_config
import pandas as pd

config = ModelConfig(development_config="config_forge.yaml")
mlflow_config = mlflow.models.ModelConfig(development_config="config_forge.yaml")
maud_config = parse_config(mlflow_config)

# data
CATALOG = config.get("data").get("catalog")
SCHEMA = config.get("data").get("schema")
CHUNKS_TABLE_NAME = config.get("data").get("chunks_table_name")
PROCESSED_DOCS_VOL = config.get("data").get("processed_docs_vol")

# retriever
VS_ENDPOINT = config.get("retriever").get("endpoint_name")
INDEX_NAME = config.get("retriever").get("index_name")
EMBEDDING_MODEL = config.get("retriever").get("embedding_model")
NUM_RESULTS = config.get("retriever").get("num_results", 5)
QUERY_TYPE = config.get("retriever").get("search_type", "hybrid")
KEY = config.get("retriever").get("primary_key")
TEXT_COL = config.get("retriever").get("text_column")

In [0]:
chunk_df = pd.read_parquet(
    f"/Volumes/{CATALOG}/{SCHEMA}/{PROCESSED_DOCS_VOL}/chunks.parquet"
)

In [0]:
def cast_to_str_array(arr):
    if arr is None:
        return []
    # Convert all elements to string, skip None values
    return [str(x) for x in arr if x is not None]

In [0]:
chunk_df_no_tables = chunk_df.drop(columns=["tables"])

In [0]:
chunk_sp = spark.createDataFrame(chunk_df_no_tables)

In [0]:
chunk_sp

In [0]:
from pyspark.sql.functions import monotonically_increasing_id

chunk_sp = spark.createDataFrame(chunk_df_no_tables)
chunk_sp = chunk_sp.withColumn("id", monotonically_increasing_id())
# chunk_sp.write.option("mergeSchema", "true").mode("overwrite").saveAsTable(f"{CATALOG}.{SCHEMA}.chunks")
display(chunk_sp.limit(5))

In [0]:
chunk_sp.write.option("mergeSchema", "true").mode("overwrite").saveAsTable(
    f"{CATALOG}.{SCHEMA}.chunks"
)

In [0]:
spark.sql(
    f"""
    ALTER TABLE {CATALOG}.{SCHEMA}.{CHUNKS_TABLE_NAME}
    ADD COLUMNS (tables ARRAY<STRING>)
"""
)

In [0]:
%sql
ALTER TABLE devanshu_pandey.multimodal.chunks
    ADD COLUMNS (tables ARRAY<STRING>)

In [0]:
%sql
select * from devanshu_pandey.multimodal.chunks

## Create Vector Search Index
Now that we have a single table for vector search, let's load it into Databricks Vector Search. There is more we can do here, but for now we simply want to

In [0]:
spark.sql(
    f"""
    ALTER TABLE {CATALOG}.{SCHEMA}.{CHUNKS_TABLE_NAME}
    SET TBLPROPERTIES (delta.enableChangeDataFeed = true)
    """
)

We pull the configuration from the agent configuration. The most important part of the vector index setup is the columns we are going to sync - in order to do filtering and advanced retrieval, we need to make sure those columns are available. If the index exists already, we will run a sync on the table. If not, we will use the SDK to create the index

In [0]:
def index_exists(client, vs_endpoint, index_name):
    try:
        client.get_index(vs_endpoint, index_name)
        return True
    except Exception as e:
        if "IndexNotFoundException" in str(e):
            return False
        else:
            raise e

In [0]:
from databricks.vector_search.client import VectorSearchClient

client = VectorSearchClient()
try:
    index = client.get_index(VS_ENDPOINT, INDEX_NAME)
    index.sync()
except:
    index = client.create_delta_sync_index(
        endpoint_name=VS_ENDPOINT,
        source_table_name=f"{CATALOG}.{SCHEMA}.{CHUNKS_TABLE_NAME}",
        index_name=f"{CATALOG}.{SCHEMA}.{INDEX_NAME}",
        pipeline_type="TRIGGERED",
        primary_key=KEY,
        embedding_source_column=TEXT_COL,
        embedding_model_endpoint_name=EMBEDDING_MODEL,
    )

## Create Vector Search As Tool

In [0]:
query = "How do I create a new layer?"

spark.sql(
    f"""
SELECT *
FROM vector_search(
  index=>'{CATALOG}.{SCHEMA}.{INDEX_NAME}',
  query_text=>'{query}',
  num_results=>{NUM_RESULTS},
  query_type=>'{QUERY_TYPE}'
)
"""
).display()

In [0]:
spark.sql(
    f"""
CREATE OR REPLACE FUNCTION {CATALOG}.{SCHEMA}.search(
  query STRING COMMENT 'A query that should resemble a section of a technical document and have at least 20 words in it'
)
RETURNS TABLE (
  matching_descriptions STRING
)
COMMENT 'A vector search of technical documents. It also includes the heading, filename and pages where the chunk appeared'
RETURN
SELECT CONCAT(
  'Filename: ', filename,
  '\n Pages: ', CAST(pages AS string),
  '\n Type: ', chunk_type,
  '\n Text: ', enriched_text
) as result
FROM vector_search(
  index=>'{CATALOG}.{SCHEMA}.{INDEX_NAME}',
  query_text=>query,
  num_results=>{NUM_RESULTS},
  query_type=>'{QUERY_TYPE}'
)
"""
)

In [0]:
spark.sql(
    f"""
  SELECT *
  FROM {CATALOG}.{SCHEMA}.search('{query}')
"""
).display()

## Chain
Setup our components and nodes

In [0]:
# API Interfaces
from src.agent.retrievers import get_vector_retriever
from databricks_langchain import ChatDatabricks

retriever = get_vector_retriever(maud_config)
model = ChatDatabricks(endpoint=maud_config.model.endpoint_name)

# Nodes
from src.agent.states import get_state
from src.agent.nodes import (
    make_query_vector_database_node,
    make_context_generation_node,
)

state = get_state(maud_config)
retriever_node = make_query_vector_database_node(retriever, maud_config)
context_generation_node = make_context_generation_node(model, maud_config)

Setup the Graph

In [0]:
# Graph
from langgraph.graph import StateGraph, START, END
from langchain_core.runnables import RunnableLambda
from src.agent.utils import graph_state_to_chat_type

workflow = StateGraph(state)
workflow.add_node("retrieve", retriever_node)
workflow.add_node("generate_w_context", context_generation_node)
workflow.add_edge(START, "retrieve")
workflow.add_edge("retrieve", "generate_w_context")
workflow.add_edge("generate_w_context", END)
app = workflow.compile()

In [0]:
input_example = {"messages": [{"role": "user", "content": "How do I add a new layer?"}]}

We can now predict using the compiled graph

In [0]:
result = app.invoke(input_example)

We can also use the graph in streaming mode

In [0]:
for msg in app.stream(input_example, stream_mode="updates"):
    print(msg)

Test our LangGraph Agent

In [0]:
import sys

sys.path.append(".")
from agent_code import AGENT

In [0]:
AGENT.predict(input_example)

In [0]:
for msg in AGENT.predict_stream(input_example):
    print(msg)

## Log The Model
This is where the deployment magic happens. This may seem a little involved, but there is a lot of magic happening:

- We set a retriever schema for MLFLow so that it can trace properly
- We set a well defined signature so that MLFLow knows that we can use the agent evaluation framework
- We provide a list of resources to allow flow through authentication
- We get a list of packages that matches of development package versions

In [0]:
maud_config.agent.experiment_location

In [0]:
experiment = mlflow.get_experiment_by_name(
    "/Users/scott.mckean@databricks.com/agent_langgraph"
)
experiment.experiment_id

In [0]:
# Setup tracking and registry
mlflow.set_tracking_uri("databricks")
mlflow.set_registry_uri("databricks-uc")
mlflow.set_experiment(maud_config.agent.experiment_location)

# Setup retriever schema
mlflow.models.set_retriever_schema(
    primary_key=maud_config.retriever.primary_key,
    text_column=maud_config.retriever.text_column,
    doc_uri=maud_config.retriever.document_uri,
)

# Setup passthrough resources
from mlflow.models.resources import (
    DatabricksVectorSearchIndex,
    DatabricksServingEndpoint,
)

databricks_resources = [
    DatabricksServingEndpoint(endpoint_name=maud_config.model.endpoint_name),
    DatabricksVectorSearchIndex(
        index_name=f"{maud_config.data.uc_catalog}.{maud_config.data.uc_schema}.{maud_config.retriever.index_name}"
    ),
]

# Get dependencies
import tomllib

with open("pyproject.toml", "rb") as f:
    toml = tomllib.load(f)
dependencies = toml["project"]["dependencies"]

With all that in place, we make our Unity Catalog enabled MLFLow logging call. This does a couple things:

- Uses code as the model to avoid serialization issues
- Passes in the 'maud' directory for our custom code
- Registers the model in Unity Catalog
- Provides an input example and signatures
- Provides the pass through resources

In [0]:
# Log the model
logged_agent_info = mlflow.pyfunc.log_model(
    name="multimodal_agent",
    python_model="agent_code.py",
    model_config="config_forge.yaml",
    code_paths=["src"],
    pip_requirements=dependencies,
    registered_model_name=maud_config.agent.uc_model_name,
    input_example=input_example,
    resources=databricks_resources,
)

print(f"Model logged and registered with URI: {logged_agent_info.model_uri}")

In [0]:
mlflow.search_experiments("attributes.")

In [0]:
# Log the prompts
prompt = mlflow.genai.register_prompt(
    name="shm.pid.retriever",
    template=maud_config.retriever.chunk_template,
)

Let's test the reloaded model before deploying it to ensure the inference works.

In [0]:
reloaded = mlflow.pyfunc.load_model(
    f"models:/{maud_config.agent.uc_model_name}/{logged_agent_info.registered_model_version}"
)
result = reloaded.predict(input_example)

## Deploy
Now we deploy the model using the Mosaic AI Agents Framework. This provides some nice convenience features out of the box:
- A review app
- Integration with playground
- Inference tables & monitoring
- Versioned deployments

In [0]:
from mlflow.deployments import get_deploy_client
from databricks import agents

client = get_deploy_client("databricks")

deployment_info = agents.deploy(
    maud_config.agent.uc_model_name,
    logged_agent_info.registered_model_version,
    scale_to_zero=True,
)

## How Databricks Apps Work

Databricks apps are a hosted container service. They use a yaml configuration file (`maud\interface\app.yaml`) to specify a run path where we run a main python file. This pattern is universal for pretty much all front ends written in python. In this accelerator we leverage Gradio, specifically the `ChatInterface` object with multimodal mode to share the retrieved images directly in chat. This is highly customizable using general UI frameworks like Streamlit, Gradio, or Flask.

The code below uses the Databricks CLI to create and deploy the app. This pattern is useful because it can be replicated in continous integration / continous deployment (CI/CD) systems.

## How to work with a multimodal chat interface

We use our Databricks App as a front end for a chat interface. This a nice abstraction for connecting to an agent, like the one we designed using LangGraph in inference and in `maud/agent`. At the core of this abstraction is an API call to the serving endpoint that is hosting the agent.

The pattern for this works like so:

UI --> [Message] --> API --> [Message] --> UI

But when dealing with detailed document retrieval, we need to ensure that we are passing images of the pages, tables, and pictures back to the UI, along with the LLM summary based on the augmented context.

## Testing Our Deployed Agent
A serving endpoint abstracts away a lot of the complexities of AI systems - especially when combined with agentic frameworks. We can test our serving endpoint below to see how the input and output signatures will react with our deployed agent.

Because most interfaces work on an API call, we use the requests package. One of the most important parts of testing the endpoint is determining the output content so we can parse it accordingly.

In [0]:
input_example = {
    "messages": [
        {"role": "user", "content": "What are the depths of the wells in the project?"}
    ]
}

In [0]:
from mlflow.deployments import get_deploy_client

result = get_deploy_client("databricks").predict(
    endpoint="agents_shm-multimodal-agent_langgraph", inputs=input_example
)

In [0]:
result